# CURE-Rec — run variations notebook

This notebook is the complete experiment control surface. Each run variation is isolated in its own commented cell so you can execute exactly the experiment you intend without editing the package code.

Run order:

1. run the end-to-end quick workflow once;
2. run the controlled regime suite;
3. calibrate/tune any failing regime;
4. run five-seed stabilization;
5. only then run 20-seed full experiments.

Do not activate all expensive cells at once.

## 1. Setup

Install once from `paper-ideas/CURE-Rec/code/`:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[dev]'
jupyter lab notebooks/00_cure_rec_quickstart.ipynb
```

In [46]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook from the CURE-Rec code directory or repository root.')

# VS Code kernels are long-lived: force local source and clear stale package modules.
sys.path[:] = [str(ROOT), *[entry for entry in sys.path if entry != str(ROOT)]]
for module_name in list(sys.modules):
    if module_name == 'cure_rec' or module_name.startswith('cure_rec.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

from cure_rec.analysis import analyze_dataset
from cure_rec.config import load_settings
from cure_rec.data import DatasetLoadResult, audit_interactions, load_dataset
from cure_rec.experiments import run_all_variations, run_seed_sweep
from cure_rec.observability import RunLogger
from cure_rec.regimes import run_regime_suite
from cure_rec.workflow import run_full_workflow

import cure_rec.data as data_layer
assert hasattr(data_layer, 'DatasetLoadResult'), f'Stale data module loaded: {data_layer.__file__}'
print('Project root:', ROOT)
print('Data module:', data_layer.__file__)
print('CURE-Rec data layer: current')


Project root: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code
Data module: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/cure_rec/data.py
CURE-Rec data layer: current


## 2. Shared configuration

All variation cells use these locations. Data download is always explicit. MovieLens and Coat can be downloaded by the loader; Yahoo! R3 must be obtained manually under its access terms.

In [47]:
DATA_ROOT = ROOT / 'data' / 'raw'
RUN_ROOT = ROOT / 'runs'

# Reuse these for public/local data variations.
MOVIELENS_SOURCE = DATA_ROOT / 'movielens_1m'
COAT_SOURCE = DATA_ROOT / 'coat'
YAHOO_SOURCE = DATA_ROOT / 'yahoo_r3'
LOCAL_CSV = None  # Set to Path('/path/to/interactions.csv') for the generic CSV variation.

def settings_for(mode: str, run_name: str):
    if mode not in {'quick', 'full'}:
        raise ValueError("mode must be 'quick' or 'full'")
    config_name = 'curesim_quickstart.yaml' if mode == 'quick' else 'curesim_full.yaml'
    settings = load_settings(ROOT / 'configs' / config_name)
    settings.run.name = run_name
    settings.run.output_root = RUN_ROOT
    return settings

print('Data root:', DATA_ROOT)
print('Run root:', RUN_ROOT)


Data root: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/data/raw
Run root: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/runs


## 3A. Data-only variation — MovieLens-1M

Fetch, standardize, audit, profile, and train registered CPU recommendation baselines. This is **not** the CURE-Sim causal run.

Set the switch to `True` only when you want to rerun external model analysis.

In [48]:
RUN_ML1M_ANALYSIS = False

if RUN_ML1M_ANALYSIS:
    ml1m = load_dataset('movielens_1m', MOVIELENS_SOURCE, download=True)
    ml1m_audit = audit_interactions(ml1m.interactions)
    ml1m_analysis = analyze_dataset(
        ml1m,
        output_root=RUN_ROOT,
        run_bpr=True,
        bpr_updates=500_000,
        max_eval_users=1_000,
        seed=42,
    )
    print('Analysis run:', ml1m_analysis.run_dir)
    print('Evidence level:', ml1m_audit.permitted_claim)
    display(ml1m_analysis.summary)
    display(ml1m_analysis.model_metrics)
else:
    print('MovieLens analysis disabled.')


MovieLens analysis disabled.


## 3B. Data-only variation — Coat

Coat is useful for randomized-vs-biased short-horizon estimator checks. It has no event timestamps or complete policy logs, so the audit should prevent long-horizon claims.

In [49]:
RUN_COAT_ANALYSIS = False

if RUN_COAT_ANALYSIS:
    coat = load_dataset('coat', COAT_SOURCE, download=True)
    coat_audit = audit_interactions(coat.interactions)
    coat_analysis = analyze_dataset(coat, output_root=RUN_ROOT, run_bpr=False)
    print('Analysis run:', coat_analysis.run_dir)
    print('Evidence level:', coat_audit.permitted_claim)
    display(coat_analysis.summary)
else:
    print('Coat analysis disabled.')


Coat analysis disabled.


## 3C. Data-only variation — Yahoo! R3 or a local CSV

Yahoo! R3 must already be available locally. The generic CSV variation is intentionally conservative and will report the strongest evidence level supported by its fields.

In [50]:
RUN_YAHOO_ANALYSIS = False
RUN_LOCAL_CSV_ANALYSIS = False

if RUN_YAHOO_ANALYSIS:
    yahoo = load_dataset('yahoo_r3', YAHOO_SOURCE)
    yahoo_analysis = analyze_dataset(yahoo, output_root=RUN_ROOT, run_bpr=False)
    print('Yahoo! R3 analysis:', yahoo_analysis.run_dir)
    print('Evidence level:', yahoo_analysis.audit.permitted_claim)

if RUN_LOCAL_CSV_ANALYSIS:
    if LOCAL_CSV is None:
        raise ValueError('Set LOCAL_CSV before enabling RUN_LOCAL_CSV_ANALYSIS.')
    local = load_dataset('csv', LOCAL_CSV)
    local_analysis = analyze_dataset(local, output_root=RUN_ROOT, run_bpr=True, bpr_updates=500_000)
    print('Local CSV analysis:', local_analysis.run_dir)
    print('Evidence level:', local_analysis.audit.permitted_claim)

if not (RUN_YAHOO_ANALYSIS or RUN_LOCAL_CSV_ANALYSIS):
    print('Yahoo! R3 and local CSV variations disabled.')


Yahoo! R3 and local CSV variations disabled.


## 4. Run all self-contained CURE-Rec variations

This master cell runs every self-contained CURE-Rec variation in the required order: external data/model analysis, quick single run, controlled regimes, quick five-seed sweep, full single run, full five-seed sweep, and full 20-seed sweep. It can run for several hours.

To prevent accidental execution, it requires both `RUN_ALL_VARIATIONS = True` and `CONFIRM_RUN_ALL = "RUN_ALL"`.


In [ ]:
# WARNING: this executes the complete experimental plan and can take hours.
RUN_ALL_VARIATIONS = False
CONFIRM_RUN_ALL = ""  # Set exactly to "RUN_ALL" only when ready.

if RUN_ALL_VARIATIONS:
    if CONFIRM_RUN_ALL != "RUN_ALL":
        raise RuntimeError("Set CONFIRM_RUN_ALL = 'RUN_ALL' to start all variations.")
    all_variations = run_all_variations(
        settings_for("quick", "curesim-all-quick"),
        settings_for("full", "curesim-all-full"),
        dataset="movielens_1m",
        source=MOVIELENS_SOURCE,
        download=True,
        bpr_updates=1_500_000,
        max_eval_users=1_000,
        quick_seeds=[42, 43, 44, 45, 46],
        full_seeds=[42, 43, 44, 45, 46],
        final_seeds=list(range(100, 120)),
    )
    print("All-variations master run:", all_variations.run_dir)
    display(all_variations.variation_summary)
else:
    print("Master all-variations run disabled. Enable only after reviewing the individual variation cells.")


## 4A. End-to-end quick variation — default first run

This is the recommended first execution. It fetches MovieLens if needed, audits and analyzes it, then runs the quick CURE-Sim causal workflow with all 64 coalitions, all quick scenarios, robust selection, and numbered assets.

In [51]:
RUN_END_TO_END_QUICK = not RUN_ALL_VARIATIONS

if RUN_END_TO_END_QUICK:
    quick_settings = settings_for('quick', 'curesim-quick-notebook')
    quick_workflow = run_full_workflow(
        quick_settings,
        dataset='movielens_1m',
        source=MOVIELENS_SOURCE,
        download=True,
        run_bpr=True,
        bpr_updates=500_000,
        max_eval_users=1_000,
    )
    ACTIVE_WORKFLOW = quick_workflow
    print('External-data analysis:', quick_workflow.analysis.run_dir)
    print('CURE run:', quick_workflow.cure_run_dir)
    print('Decision:', quick_workflow.decision.status.value, quick_workflow.decision.selected_interventions)
else:
    print('Quick end-to-end variation disabled.')


2026-08-04 19:54:04,185 | INFO | run_started | {"config_hash": "a8c97790372307eb", "run_id": "curesim-quick-notebook-20260804T185404Z-cae4cb77"}
2026-08-04 19:54:04,186 | INFO | exact_game_started | {}
2026-08-04 19:54:04,187 | INFO | simulator_ready | {"horizon": 4, "n_items": 72, "n_users": 24, "scenario": "nominal"}
2026-08-04 19:54:09,572 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.4210904357912536, "scenario": "nominal", "shapley_efficiency_gap": 5.551115123125783e-17}
2026-08-04 19:54:09,573 | INFO | simulator_ready | {"horizon": 4, "n_items": 72, "n_users": 24, "scenario": "fatigue_stress"}
2026-08-04 19:54:14,923 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.4187732306444988, "scenario": "fatigue_stress", "shapley_efficiency_gap": 5.551115123125783e-17}
2026-08-04 19:54:14,923 | INFO | simulator_ready | {"horizon": 4, "n_items": 72, "n_users": 24, "scenario": "popularity_stress"}
2026-08-04 19:54:20,269 | INFO | scenario_game_co

## 4B. End-to-end full variation — larger behavioural CURE-Sim

This variation is computationally expensive. It runs 120 users, 240 items, a 12-step horizon, four scenarios, exact 64-coalition evaluation, vectorized BPR-MF, and all assets. Enable only after the quick variation and regime suite are healthy.

In [52]:
RUN_END_TO_END_FULL = False

if RUN_END_TO_END_FULL:
    full_settings = settings_for('full', 'curesim-full-notebook')
    full_workflow = run_full_workflow(
        full_settings,
        dataset='movielens_1m',
        source=MOVIELENS_SOURCE,
        download=True,
        run_bpr=True,
        bpr_updates=1_500_000,
        max_eval_users=1_000,
    )
    ACTIVE_WORKFLOW = full_workflow
    print('External-data analysis:', full_workflow.analysis.run_dir)
    print('CURE run:', full_workflow.cure_run_dir)
    print('Decision:', full_workflow.decision.status.value, full_workflow.decision.selected_interventions)
else:
    print('Full end-to-end variation disabled.')


Full end-to-end variation disabled.


## 5. Inspect the latest end-to-end workflow

This cell works after either quick or full end-to-end variation. It separates external-data baseline analysis from the causal CURE-Sim result.

In [53]:
if 'ACTIVE_WORKFLOW' not in globals():
    print('Run variation 4A or 4B first.')
else:
    workflow = ACTIVE_WORKFLOW
    print('External-data evidence level:', workflow.analysis.audit.permitted_claim)
    display(workflow.analysis.summary)
    display(workflow.analysis.model_metrics)

    game = workflow.game
    RUN_DIR = workflow.cure_run_dir
    decision = workflow.decision
    display(game.regions.sort_values('phi_mean', ascending=False))
    display(game.interaction_table.sort_values('interaction_mean', ascending=False))

    coalitions = game.coalition_table.groupby('mask', as_index=False).agg(
        lower_improvement=('improvement', 'min'),
        upper_improvement=('improvement', 'max'),
        cost=('cost', 'first'),
        interventions=('active_interventions', 'first'),
    ).sort_values('lower_improvement', ascending=False)
    display(coalitions.head(12))


External-data evidence level: descriptive_or_semisynthetic


,dataset,users,items,interactions,positive_interactions,positive_rate,density,timestamps_available
0,movielens_1m,6040,3706,1000209,575281,0.575161,0.044684,True


,model,evaluated_users,recall_at_k,ndcg_at_k,hit_rate_at_k
0,popularity,1000,0.049,0.025520,0.049
1,bpr_mf,1000,0.037,0.016912,0.037


,intervention,phi_lower,phi_upper,phi_mean,psi_feasible_lower,psi_feasible_upper,phi_psi_sign_agree
0,repeat_cap,0.019328,0.021651,0.020355,0.019107,0.021458,True
3,diversify,-0.058360,-0.057684,-0.057936,-0.057792,-0.056812,True
2,tail_slot,-0.079257,-0.078018,-0.078605,-0.080382,-0.078750,True
4,novel_slot,-0.092236,-0.087646,-0.089787,-0.093722,-0.088783,True
1,explore_slot,-0.099465,-0.098286,-0.098737,-0.100854,-0.099472,True
5,provider_balance,-0.116113,-0.114098,-0.115354,-0.116218,-0.114792,True


,intervention_i,intervention_j,interaction_lower,interaction_upper,interaction_mean
14,novel_slot,provider_balance,0.002719,0.004190,0.003328
12,diversify,novel_slot,0.002373,0.003395,0.002879
5,explore_slot,tail_slot,0.001386,0.003722,0.002804
7,explore_slot,novel_slot,0.002158,0.003382,0.002772
3,repeat_cap,novel_slot,0.000554,0.001605,0.001177
6,explore_slot,diversify,0.000240,0.001268,0.000872
10,tail_slot,novel_slot,-0.000909,0.001235,0.000504
9,tail_slot,diversify,-0.000561,0.000884,0.000109
2,repeat_cap,diversify,-0.000787,-0.000456,-0.000576
8,explore_slot,provider_balance,-0.001096,-0.000162,-0.000596


,mask,lower_improvement,upper_improvement,cost,interventions
1,1,0.027033,0.032587,0.05,repeat_cap
0,0,0.000000,0.000000,0.00,
9,9,-0.032752,-0.027082,0.11,repeat_cap;diversify
5,5,-0.055457,-0.052778,0.13,repeat_cap;tail_slot
8,8,-0.060242,-0.058317,0.06,diversify
17,17,-0.067962,-0.064039,0.13,repeat_cap;novel_slot
4,4,-0.077633,-0.073657,0.08,tail_slot
3,3,-0.079463,-0.073913,0.15,repeat_cap;explore_slot
33,33,-0.092927,-0.089431,0.17,repeat_cap;provider_balance
16,16,-0.096463,-0.092377,0.08,novel_slot


## 6. Controlled cooperative-structure regime suite

Run this before large behavioural seed sweeps. It verifies that the exact game, Shapley values, interaction values, and planner modes recover known additive, complementary, redundant, antagonistic, delayed, repair, and misspecified structures.

In [54]:
RUN_REGIME_SUITE = False

if RUN_REGIME_SUITE:
    regime_settings = settings_for('quick', 'curesim-regime-suite')
    regime_logger = RunLogger(regime_settings)
    try:
        regime_suite = run_regime_suite(regime_settings, regime_logger)
        regime_logger.close(status='completed')
    except Exception:
        regime_logger.close(status='failed')
        raise
    print('Regime-suite assets:', regime_suite.run_dir)
    display(regime_suite.summary)
    display(regime_suite.attribution_recovery.groupby('regime', as_index=False).agg(
        shapley_mae=('absolute_error', 'mean'),
        sign_accuracy=('sign_correct', 'mean'),
    ))
else:
    print('Regime suite disabled.')


Regime suite disabled.


## 7A. Paired quick five-seed stabilization variation

This is the recommended first stochastic stability run after the regime suite passes. Common random numbers are shared within every seed across coalitions.

In [55]:
RUN_QUICK_FIVE_SEEDS = False
QUICK_SEEDS = [42, 43, 44, 45, 46]

if RUN_QUICK_FIVE_SEEDS:
    quick_sweep_settings = settings_for('quick', 'curesim-quick-sweep')
    quick_sweep = run_seed_sweep(quick_sweep_settings, QUICK_SEEDS)
    print('Quick seed sweep:', quick_sweep.run_dir)
    display(quick_sweep.decisions)
    display(quick_sweep.attributions.groupby('intervention', as_index=False).agg(
        phi_mean=('phi_mean', 'mean'),
        phi_std=('phi_mean', 'std'),
        positive_rate=('phi_lower', lambda x: float((x > 0).mean())),
    ))
else:
    print('Quick five-seed sweep disabled.')


Quick five-seed sweep disabled.


## 7B. Paired full five-seed stabilization variation

Run only after quick stabilization. This is the direct precursor to final multi-seed analysis and may take roughly an hour on the full configuration.

In [56]:
RUN_FULL_FIVE_SEEDS = False
FULL_STABILIZATION_SEEDS = [42, 43, 44, 45, 46]

if RUN_FULL_FIVE_SEEDS:
    full_sweep_settings = settings_for('full', 'curesim-full-sweep')
    full_sweep = run_seed_sweep(full_sweep_settings, FULL_STABILIZATION_SEEDS)
    print('Full five-seed sweep:', full_sweep.run_dir)
    display(full_sweep.decisions)
    display(full_sweep.attributions.groupby('intervention', as_index=False).agg(
        phi_mean=('phi_mean', 'mean'),
        phi_std=('phi_mean', 'std'),
        positive_rate=('phi_lower', lambda x: float((x > 0).mean())),
    ))
else:
    print('Full five-seed sweep disabled.')


Full five-seed sweep disabled.


## 7C. Paired full 20-seed final variation

Do not run until all controlled regimes recover their expected structures and the five-seed full sweep is stable. This is the final statistical experiment variation and may take several hours.

In [57]:
RUN_FULL_TWENTY_SEEDS = False
FINAL_SEEDS = list(range(100, 120))

if RUN_FULL_TWENTY_SEEDS:
    final_settings = settings_for('full', 'curesim-final-sweep')
    final_sweep = run_seed_sweep(final_settings, FINAL_SEEDS)
    print('Full 20-seed sweep:', final_sweep.run_dir)
    display(final_sweep.decisions)
    display(final_sweep.attributions.groupby('intervention', as_index=False).agg(
        phi_mean=('phi_mean', 'mean'),
        phi_std=('phi_mean', 'std'),
        positive_rate=('phi_lower', lambda x: float((x > 0).mean())),
    ))
else:
    print('Full 20-seed sweep disabled.')


Full 20-seed sweep disabled.


## 8. Inspect all generated assets and logs

Run after variation 4A or 4B. The registry distinguishes generated CURE-Sim assets from manual literature work and future audited real-log OPE assets.

In [58]:
if 'RUN_DIR' not in globals():
    print('Run quick or full end-to-end variation first.')
else:
    asset_manifest = json.loads((RUN_DIR / 'artifacts' / 'asset_manifest.json').read_text())
    display(pd.DataFrame(asset_manifest))

    print('Generated CURE tables:')
    for path in sorted((RUN_DIR / 'tables').glob('*.csv')):
        print('-', path.name)
    print('\nGenerated CURE figures:')
    for path in sorted((RUN_DIR / 'figures').glob('*.png')):
        print('-', path.name)

    events = pd.DataFrame([json.loads(line) for line in (RUN_DIR / 'logs' / 'events.jsonl').read_text().splitlines()])
    display(events[['timestamp_utc', 'event']].tail(20))
    display(json.loads((RUN_DIR / 'artifacts' / 'explanation_card.json').read_text()))


,exists,id,path,purpose,scope
0,True,Table 1,tables/table_01_asset_registry.csv,"Asset provenance, scope, and readiness",generated
1,True,Table 2,tables/table_02_benchmark_configuration.csv,CURE-Sim and policy configuration,generated
2,True,Table 3,tables/table_03_attribution_regions.csv,Full-game Shapley and feasibility-aware semiva...,generated
3,True,Table 4,tables/table_04_uncertainty_summary.csv,Scenario uncertainty widths and attribution signs,generated
4,True,Table 5,tables/table_05_portfolio_decision.csv,Robust selected portfolio and constraint diagn...,generated
5,True,Table 6,tables/table_06_long_term_tradeoffs.csv,Base versus selected policy outcomes by scenario,generated
6,True,Table 7,tables/table_07_selection_comparison.csv,"Base, best-single, full, and robust portfolio ...",generated
7,True,Table 8,tables/table_08_runtime_summary.csv,Coalition evaluation runtime by scenario and c...,generated
8,True,Figure 1,figures/figure_01_framework.png,CURE-Rec execution flow,generated
9,True,Figure 2,figures/figure_02_shapley_regions.png,Shapley regions and selected interventions,generated


Generated CURE tables:
- coalition_values.csv
- interaction_regions.csv
- shapley_regions.csv
- table_01_asset_registry.csv
- table_02_benchmark_configuration.csv
- table_03_attribution_regions.csv
- table_04_uncertainty_summary.csv
- table_05_portfolio_decision.csv
- table_06_long_term_tradeoffs.csv
- table_07_selection_comparison.csv
- table_08_runtime_summary.csv

Generated CURE figures:
- figure_01_framework.png
- figure_02_shapley_regions.png
- figure_03_uncertainty_widths.png
- figure_04_interaction_heatmap.png
- figure_05_trajectory_comparison.png
- figure_06_decision_card.png
- figure_07_runtime_by_cardinality.png
- figure_08_scenario_sensitivity.png


,timestamp_utc,event
392,2026-08-04T18:54:20.279314+00:00,exact_game_completed
393,2026-08-04T18:54:20.279689+00:00,robust_planning_started
394,2026-08-04T18:54:20.280202+00:00,planner_mode_resolved
395,2026-08-04T18:54:20.280810+00:00,portfolio_rejected
396,2026-08-04T18:54:20.281317+00:00,portfolio_rejected
397,2026-08-04T18:54:20.281839+00:00,portfolio_rejected
398,2026-08-04T18:54:20.282296+00:00,portfolio_rejected
399,2026-08-04T18:54:20.283176+00:00,portfolio_rejected
400,2026-08-04T18:54:20.283618+00:00,portfolio_rejected
401,2026-08-04T18:54:20.283963+00:00,portfolio_rejected


{'decision': {'action': 'improve_selected',
  'base_feasible': True,
  'cost': 0.05,
  'fatigue_upper': 0.0,
  'feasible': True,
  'lower_improvement': 0.02703258221131133,
  'mode': 'improvement',
  'provider_disparity_upper': 0.22428385416666674,
  'reason': 'Base policy is feasible; selected the exact maximin-improvement portfolio.',
  'relevance_delta_lower': -0.03932083801222208,
  'selected_interventions': ['repeat_cap'],
  'selected_mask': 1,
  'status': 'improve_selected',
  'upper_improvement': 0.03258652756926306},
 'interpretation': {'improvement_mode': 'A feasible base policy may be retained when no portfolio robustly improves it.',
  'positive_certificate': 'phi_lower > 0 means positive order-averaged marginal contribution across configured scenarios.',
  'repair_mode': 'An infeasible base policy cannot be retained; select a feasible repair or certify no feasible portfolio.',
  'selection_rule': 'Portfolio selection used direct robust improvement, not summed Shapley lower 

## 10. Scientific execution order

- **Individual mode:** run 4A once, then 6, then 7A; continue to 4B, 7B, and 7C only after each gate passes.
- **Master mode:** set `RUN_ALL_VARIATIONS = True` and `CONFIRM_RUN_ALL = "RUN_ALL"` in Section 4. The orchestrator preserves this same order automatically.

External MovieLens analysis is useful for model/data robustness. The complete causal claims remain grounded in the controlled regime suite and CURE-Sim behavioural experiments.